# Catching a Bit Flip

This notebook reconstructs the middle-qubit bit-flip experiment documented in Sajal Farooq's internship presentation. It runs locally in an ideal state-vector simulation using Python's standard library.

The wire order is **D0 D1 D2 | A0 A1**. D0–D2 are data qubits. A0 stores D0 XOR D1, and A1 stores D1 XOR D2. The circuit uses the ancillas as coherent controls for a Toffoli gate on D1. The displayed checkpoints inspect the simulated state without inserting measurement gates.

In [1]:
from itertools import product
from math import isclose, sqrt

# Use tuples in diagram order so basis labels read left to right.
def controlled_x(state, target, *controls):
    updated = {}
    for bits, amplitude in state.items():
        result = list(bits)
        if all(bits[control] for control in controls):
            result[target] ^= 1
        updated[tuple(result)] = amplitude
    return updated


def prepare_input(alpha=0, beta=1):
    return {
        (bit, 0, 0, 0, 0): complex(amplitude)
        for bit, amplitude in enumerate((alpha, beta))
        if amplitude
    }


def encode(state):
    return controlled_x(controlled_x(state, 1, 0), 2, 0)


def extract_syndrome(state):
    for data, ancilla in [(0, 3), (1, 3), (1, 4), (2, 4)]:
        state = controlled_x(state, ancilla, data)
    return state


def correct_d1(state):
    return controlled_x(state, 1, 3, 4)


def basis_label(bits):
    data = "".join(map(str, bits[:3]))
    ancillas = "".join(map(str, bits[3:]))
    return f"|{data}⟩|{ancillas}⟩"

## Reproduce the documented run

Prepare D0 in |1⟩ and encode it as |111⟩. Inject an X on D1 to produce |101⟩. The two parity checks both return 1, so the ancilla pair activates the Toffoli and restores the data to |111⟩.

In [2]:
encoded = encode(prepare_input())
flipped = controlled_x(encoded, 1)
detected = extract_syndrome(flipped)
corrected = correct_d1(detected)

checkpoints = [
    ("After encoding", encoded, (1, 1, 1, 0, 0)),
    ("After the D1 flip", flipped, (1, 0, 1, 0, 0)),
    ("After syndrome extraction", detected, (1, 0, 1, 1, 1)),
    ("After correction", corrected, (1, 1, 1, 1, 1)),
]
for stage, state, expected in checkpoints:
    assert state == {expected: 1 + 0j}, (stage, state)
    print(f"{stage:<28} {basis_label(expected)}")

print(f"Final-state probability: {abs(corrected[(1, 1, 1, 1, 1)])**2:.0%}")

After encoding               |111⟩|00⟩
After the D1 flip            |101⟩|00⟩
After syndrome extraction    |101⟩|11⟩
After correction             |111⟩|11⟩
Final-state probability: 100%


## Check the scope of the correction

The syndrome distinguishes the three single-data-qubit flips. This circuit implements recovery for syndrome 11. The following checks also verify that recovery preserves the relative phase of an encoded superposition when D1 is flipped.

In [3]:
expected_syndromes = {None: (0, 0), 0: (1, 0), 1: (1, 1), 2: (0, 1)}
for error, syndrome in expected_syndromes.items():
    test_state = encoded if error is None else controlled_x(encoded, error)
    checked = extract_syndrome(test_state)
    assert next(iter(checked))[3:] == syndrome
    label = "No error" if error is None else f"X on D{error}"
    print(f"{label:<10} → {syndrome[0]}{syndrome[1]}")

alpha, beta = 1 / sqrt(2), 1j / sqrt(2)
superposition = encode(prepare_input(alpha, beta))
recovered = correct_d1(extract_syndrome(controlled_x(superposition, 1)))
assert recovered == {(0, 0, 0, 1, 1): alpha, (1, 1, 1, 1, 1): beta}
assert isclose(sum(abs(a)**2 for a in recovered.values()), 1)
print("D1 recovery preserves both superposition amplitudes.")

No error   → 00
X on D0    → 10
X on D1    → 11
X on D2    → 01
D1 recovery preserves both superposition amplitudes.
